### Evaluation
이 예제에서는 Evaluation을 해보겠습니다. `rouge`같은 벤치마크가 아닌 LLM을 이용한 평가 방법입니다.  
물론 벤치마크 평가가 사장된 것은 아닙니다만, 현재는 LLM을 통한 평가 방법을 일반적으로 많이 사용하고 있습니다.  

이 예제는 Judging LLM-as-a-Judge 논문 ([https://arxiv.org/pdf/2306.05685](https://arxiv.org/pdf/2306.05685))과 [Openai evals](https://github.com/openai/evals)를 바탕으로 작성되었습니다.  
프롬프트 또한 논문을 기반으로 작성되었습니다.  

평가는 아래의 순서로 진행됩니다.  
1. 튜닝전 모델과 튜닝 후 모델을 불러옵니다.
2. 작성해놓은 평가 프롬프트를 input으로 하여 각각의 모델로 output을 생성합니다.
3. ChatGPT-4o 각 모델들의 output을 전달하여 평가하도록 요청합니다.

In [1]:
!pip install --quiet\
peft\
accelerate\
flash-attn\
transformers\
openai\
python-dotenv\
selenium\
colorama

In [11]:
import os
import sys
import json
import copy
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import AutoPeftModelForCausalLM
import torch

# Colab에서 실행시 아래 주석처리된 부분을 해제하세요
# from google.colab import userdata, drive
# utils, prompts 커스텀 모듈 사용을 위해 구글 드라이브에 마운트합니다.
# drive.mount('/content/drive')

# # git clone 받은 위치로 지정합니다.
# path = "/content/drive/MyDrive/instruction-tuning-with-rag-example"
# sys.path.append(path)

import utils
import prompts

### 1. 튜닝전 모델과 튜닝 후 모델을 불러옵니다

In [17]:
# colab실행시 환경변수를 userdata로 저장하여 사용하세요
# try: 
#     token = userdata.get("HF_TOKEN")
# except:
#     token = os.getenv("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    torch_dtype=torch.bfloat16,
    token=token,
    attn_implementation="flash_attention_2"
).to("cuda")

finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    "aiqwe/gemma-2b-it-example-v1",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2"
).to("cuda")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/618 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

### 2. 작성해놓은 평가 프롬프트를 input으로 하여 각각의 모델로 output을 생성합니다.

In [18]:
# 평가 데이터 불러오기
import os

# colab 인경우
# path = os.path.join(path, "data/eval_dataset.txt")
path = "data/eval_dataset.txt"

with open(path, "r") as f:
    eval_inputs = f.readlines()

eval_inputs = [inputs.strip("\n") for inputs in eval_inputs]
eval_inputs

['정비 사업의 진행 절차를 알려줘.',
 '실거주 의무가 뭔가요?',
 '신속통합기획 진행중인 곳이 어디인가요?',
 '청약에 당첨되려면 뭘 해야할까요?',
 '부동산 주요 정책은 뭐가 있나요?',
 '전세 사기에 대비하는 방법을 알려주세요.',
 '부동산 매도시 양도세에 대해 설명해주세요.',
 '전매 제한이 뭔가요?',
 '주택 담보 대출을 받으려면 어떻게 해야할까요?',
 'DSR, DTI, LTV에 대해 설명해주세요.',
 '디에이치 방배는 언제 분양되나요?',
 '한남 뉴타운 진행사항이 어떻게 될까요?',
 '분양가 상한제시 어떤 규제를 받나요?',
 '임차인 우서 변제권이 뭔가요?',
 '전세 중개수수료는 얼마나 내야하나요?']

In [19]:
# Base모델 / Fine-tuning 모델 모두 Chat Template을 사용했기 때문에 Inference때도 Chat Template을 사용해줍니다.
inputs = [tokenizer.apply_chat_template(
    conversation=[
        {"role": "user", "content": query}
    ],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=False
) for query in eval_inputs]

In [21]:
inputs

['<bos><start_of_turn>user\n정비 사업의 진행 절차를 알려줘.<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\n실거주 의무가 뭔가요?<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\n신속통합기획 진행중인 곳이 어디인가요?<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\n청약에 당첨되려면 뭘 해야할까요?<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\n부동산 주요 정책은 뭐가 있나요?<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\n전세 사기에 대비하는 방법을 알려주세요.<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\n부동산 매도시 양도세에 대해 설명해주세요.<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\n전매 제한이 뭔가요?<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\n주택 담보 대출을 받으려면 어떻게 해야할까요?<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\nDSR, DTI, LTV에 대해 설명해주세요.<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\n디에이치 방배는 언제 분양되나요?<end_of_turn>\n<start_of_turn>model\n',
 '<bos><start_of_turn>user\n한남 뉴타운 진행사항이 

In [18]:
from utils import generate

# Base 모델 / 튜닝한 모델의 output generate하기
input_text = tokenizer(inputs, return_tensors="pt", padding=True).to(base_model.device)

# base modeld에서 배치로 generate
outputs = base_model.generate(**input_text, max_new_tokens=512, repetition_penalty = 1.5, temperature=0)
decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
# output만 가져오기
base_decoded = [text.split("model")[1] for text in decoded]

# fine-tuning 모델 배치로 generate
outputs = finetuned_model.generate(**input_text, max_new_tokens=512, repetition_penalty = 1.5, temperature=0)
decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
finetuned_decoded = [text.split("model")[1] for text in decoded]

# 저장하기
eval_dataset = dict(inputs=eval_inputs, completion1=base_decoded, completion2=finetuned_decoded)
utils.jsave(data=eval_dataset, save_path=os.path.join(path, "data/eval_dataset.json"), mode="w", indent=4)

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


In [19]:
eval_dataset = utils.jload(os.path.join(path, "data/eval_dataset.json"))

In [20]:
from colorama import Fore, Style

idx = 4

print(Fore.MAGENTA + "INPUT:" + Style.RESET_ALL)
print(eval_dataset['inputs'][idx] + "\n")

print(Fore.MAGENTA + "BASE MODEL:" + Style.RESET_ALL)
print(eval_dataset['completion1'][idx] + "\n")

print(Fore.MAGENTA + "FINETUNED MODEL:" + Style.RESET_ALL)
print(eval_dataset['completion2'][idx])

INPUT:
부동산 주요 정책은 뭐가 있나요?

BASE MODEL:

**부동산 주요 정책**

* **국민주택 공급 지원:** 국민이 부동산에 거주하기 위해 필요한 다양한 수준의 저비용과 장기적인 보증을 제공합니다.
* **부동산 용도 지정:** 특정 지역에서 특정 용도로 사용할 수 있는지 명확하게 설정하여 개발 및 활용을 유도합니다.
* **부동산 관리 시스템 개선:** 효율성과 안전성을 높이기 위한 철저하고 투명한 부동산 관리 시스템 구축.
* **거래 통제:** 부동산 가격의 과도한 상승을 방지하기 위하여 거래를 제어하는 기반 마련.
* **자연스러운 개발 환경 조성:** 자원 활용을 고려한 부동산 계획, 교통, 교육 등의 시스템적 지원을 강화합니다.
* **부동산 투자 확대:** 국내외 투자자에게 부동산 투자가 더욱 안전하고 효과적으로 이루어질 수 있도록 지원을 제공합니다.
* **부동산 법률 완화:** 부동산 관련 법률을 간소화하고 효과적으로 운영하도록 도와줍니다.

FINETUNED MODEL:

현재 부동산 관련 중요한 정책으로는 다주택자에 대한 취득세 중과 완화, 임대사업 활성화 방안 도출 등이 있습니다. 특히 청년 및 신혼부부를 위한 저금리 대출 지원도 포함되어 있으며, 이러한 제도들은 실수요자들에게 큰 도움을 제공하고 있습니다. 또한, 재건축·재개발의 원활한 진행을 위해 '공공기여율' 개선 조치와 같은 법적 요소들이 마련되고 있습니다. 따라서 현재 가장 필요로 하는 것은 해당 지역에서 시설물 보존 관리 기법 강화입니다. 예를 들어, 공원이나 산림에는 유지보수 계획 수립 과정부터 현장 점검 확대까지 철저하게 준비됩니다. 전반적으로 국민들의 생활 편의를 높이고 환경 을 보호하는 데 초점을 맞춘 것입니다.



Judging LLM-as-a-Judge([https://arxiv.org/pdf/2306.05685](https://arxiv.org/pdf/2306.05685))논문에 의하면 2개의 Assistant를 비교할 때 순서가 영향을 줄 수 있다고 합니다.  
따라서 base model과 finetuned model의 순서를 바꿔서 한번 더 평가를 받습니다.  
+ 첫 번째 평가 : (A: Base Model, B: Finetuned Model)  
+ 두 번째 평가 : (A: FineTuned Model, B: Finetuned Model)  

따라서 FineTuning한 모델이 평가가 좋으려면, 첫번째 평가는 B가 많아야하고 두번째는 A가 많아야합니다.  

In [21]:
from colorama import Fore, Style

# 프롬프트 샘플
for input, out1, out2 in zip(eval_dataset['inputs'], eval_dataset['completion1'], eval_dataset['completion2']):
    print(Fore.MAGENTA + "첫번째 평가할 프롬프트" + Style.RESET_ALL)
    print(prompts.EVAL_BATTLE_PROMPT.format(question=input, answer_a=out1, answer_b=out2))
    print("\n\n")
    print(Fore.MAGENTA + "두번째 평가할 프롬프트" + Style.RESET_ALL)
    print(prompts.EVAL_BATTLE_PROMPT.format(question=input, answer_a=out2, answer_b=out1))
    break

첫번째 평가할 프롬프트

[System]
공정한 심사위원이 되어 2개의 AI 모델이 제공하는 답변의 품질을 평가하세요.
사용자의 지시를 잘 따르고, 사용자의 질문에 더 잘 답변하는 Assistant를 선택하세요.
답변의 유용성, 관련성, 정확성, 깊이, 창의성 및 세부 수준과 같은 요소를 고려하여 평가해야 합니다.
가장 중요하게 고려할 사항은 정확성입니다.
최대한 객관적으로 판단해야 하며, 어떠한 입장의 bias도 있어선 안됩니다.
답변이 제시된 순서가 판단에 영향을 미치지 않도록 해야합니다.
답변의 길이가 평가에 영향을 미치지 않도록 합니다.
특정 Assistant의 이름을 선호해서는 안됩니다.
가능한 객관적으로 평가해야합니다.

평가 후 최종 평점을 산출해야 합니다:
- A Assistant의 답변이 더 좋으면 "A", B Assistant의 답변이 더 좋으면 "B", 동점이면 "C"로 표기합니다.

평가후 JSON 포맷에따라 평점과 설명을 출력하세요.
Markdown 양식은 제거하고 출력하세요.
출력할 JSON 스키마는 아래와 같습니다.
{"selected_assistant": selected_assistant: str, "reason": reason: str}
- selected_assistant : 선택한 Assistant
- reason : selected_assistant를 선택한 이유

[사용자 질문]
정비 사업의 진행 절차를 알려줘.
[사용자 질문]
[Assistant A의 답변 시작]

**정비 사업 진행 절차**

1. **목표 설정:** 정비 사업을 수행하기 위한 목표와 기대 결과를 명확하게 설정해야 합니다.


2. **시스템 분석 및 평가:** 현재 시스템의 문제점과 기능 부족을 파악하고, 이에 대한 해결책을 도출해야 합니다.


3. **계획 마련:** 정비 계획서를 작성하여 정비 과정에서 필요한 모든 작업을 상세하게 기술해야 합니다.


4. **자원 확보:** 정비 비용을 예산으로 책정하고, 필요한 자원(인력, 장비 등)을 구축해

### 3. Base 모델과 Battle

In [9]:
openai_key = userdata.get("OPENAI_API_KEY")

scores_1 = []
scores_2 = []
for input, out1, out2 in zip(eval_dataset['inputs'], eval_dataset['completion1'], eval_dataset['completion2']):
    result = utils.get_completion(prompts.EVAL_BATTLE_PROMPT.format(question=input, answer_a=out1, answer_b=out2), api_key=openai_key, model="gpt-4o")
    scores_1.append(result)
    result = utils.get_completion(prompts.EVAL_BATTLE_PROMPT.format(question=input, answer_a=out2, answer_b=out1), api_key=openai_key, model="gpt-4o")
    scores_2.append(result)

In [13]:
# 이긴 Assistant 보기
winner1 = [json.loads(s1)['selected_assistant'] for s1 in scores_1]
winner2 = [json.loads(s2)['selected_assistant'] for s2 in scores_2]

In [22]:
import pandas as pd

winner_df = pd.DataFrame.from_dict({"eval1": winner1, "eval2": winner2})
winner_df

,eval1,eval2
0,A,B
1,B,A
2,B,A
3,B,A
4,B,B
5,B,A
6,B,A
7,A,A
8,A,B
9,B,A


In [40]:
# Finetuning 모델의 승리 갯수
wins = (winner_df['eval1'] == "B").sum() + (winner_df['eval2'] == "A").sum()
total = winner_df['eval1'].count() * 2
print(f"승률 : {wins / total:.2%}")


승률 : 73.33%


### 4. LLM에게 점수받기

In [10]:
openai_key = userdata.get("OPENAI_API_KEY")

scores = []
for input, out2 in zip(eval_dataset['inputs'], eval_dataset['completion2']):
    result = utils.get_completion(prompts.EVAL_SCORE_PROMPT.format(question=input, answer=out2), api_key=openai_key, model="gpt-4o")
    scores.append(result)

In [30]:
json.loads(scores[0])

{'score': 3,
 'reason': "답변은 정비 사업의 주요 단계를 다루고 있지만, 여러 부분에서 부정확하거나 불명확한 설명이 포함되어 있습니다. 예를 들어, '조합 설립' 단계에서 '이주민 모집'이라는 표현은 부적절하며, '소유권자 동호수 확보'라는 표현도 모호합니다. 또한, '인허가와 허가 사항 확인' 부분에서 '건축심원일 현재 시설물 보존등기 상태여야 합니다'라는 문장은 이해하기 어렵습니다. '분양 계약 체결' 부분도 구체적인 설명이 부족합니다. 전반적으로 정확성과 깊이 면에서 개선이 필요합니다."}

RLHF 학습이 되지 않았는데, 프롬프트상 RLHF를 평가하는 내용이 있어서 전반적으로 점수가 낮은 것 같습니다.

In [26]:
scores_list = [int(json.loads(s)['score']) for s in scores]

In [27]:
scores_list

[3, 2, 6, 3, 3, 3, 2, 2, 6, 2, 2, 3, 2, 1, 1]

### 4. vLLM

Benchmark를 측정하기 전에 모델은 Inference를 하게 됩니다.  
vLLM을 통해서 Inference를 실습해보도록 합니다.  

+ Paper : [https://arxiv.org/pdf/2309.06180](https://arxiv.org/pdf/2309.06180)
  
트랜스포머 기반의 Decoder 모델들은 이전 Sequence를 기반으로 Next Token을 예측합니다.  
따라서 이전 Sequence를 메모리에 저장(Key Value Cache, KV Cache)하고 다음 토큰의 Attention을 계산해야합니다.  
이 때 KV Cache가 GPU의 VRAM을 상당 부분 차지하는 것은 큰 애로사항이었습니다.  

vLLM은 가상 메모리 기법 중 물리 메모리를 고정 분할하는 Paging 기법에서 아이디어를 얻었습니다.
+ Logical KV Cache - Physical KV Cache를 맵핑(가상메모리의 맵핑 테이블)
+ Physical KV Cache는 블록단위로 관리

이를 통해 크게 2가지 장점을 얻습니다.
+ 이제 `max_len`만큼의 메모리를 예약하지 않아도 됩니다. Block 단위의 idle만 발생합니다.
+ Block 단위로 메모리 공유를 할 수 있습니다.(i.e. Input Prompt)

In [2]:
!pip install vllm --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.4.1+cu124 requires torch==2.4.1, but you have torch 2.4.0 which is incompatible.


vllm에서 LoRA 사용시,
+ base모델은 LLM 클래스로 할당하는 동시에 `enable_lora=True` 설정
+ LoRA Adapter는 generate에 추가합니다.
vllm에 [가이드](https://docs.vllm.ai/en/latest/models/lora.html)가 있으니 참조해주세요.

In [2]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

/usr/local/lib/python3.11/dist-packages/vllm/connections.py:8: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from vllm.version import __version__ as VLLM_VERSION


In [5]:
# Basemodel Load
# 모델을 로드하기전에 환경변수 HF_TOKEN="토큰"을 추가해주세요, 또는 os.environ으로 추가해주세요
import os
llm = LLM("google/gemma-2b-it", enable_lora=True)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

INFO 10-21 18:58:28 llm_engine.py:237] Initializing an LLM engine (vdev) with config: model='google/gemma-2b-it', speculative_config=None, tokenizer='google/gemma-2b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=google/gemma-2b-it, use_v2_block_manager=True, num_scheduler_steps=1, chunked_prefill_enabled=False multi_step_stream_outputs=True, enable

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 10-21 18:59:05 model_runner.py:1071] Loading model weights took 4.6720 GB
INFO 10-21 18:59:12 gpu_executor.py:122] # GPU blocks: 49109, # CPU blocks: 14563
INFO 10-21 18:59:12 gpu_executor.py:126] Maximum concurrency for 8192 tokens per request: 95.92x
INFO 10-21 18:59:16 model_runner.py:1402] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 10-21 18:59:16 model_runner.py:1406] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 10-21 18:59:42 model_runner.py:1530] Graph capturing finished in 27 secs.


Finetuning한 모델을 다운로드합니다.

In [6]:
from huggingface_hub import snapshot_download

lora_path = snapshot_download(repo_id="aiqwe/gemma-2b-it-example-v1")

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

.gitattributes:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

(…)tfevents.1714909578.2dbe07814dcb.49535.0:   0%|          | 0.00/20.1k [00:00<?, ?B/s]

...

adapter_model.safetensors:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.6k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

In [11]:
sampling_params = SamplingParams(
    temperature=0,
    max_tokens=512,
    stop=["[/assistant]"]
)

prompts = [
    "전세 계약에 대해 알려줘요.",
    "내년 부동산 전망은 어떨까요?"
]

response = llm.generate(prompts, sampling_params=sampling_params, lora_request=LoRARequest("my_model", 1, lora_path)) # (어댑터이름, 어댑터 번호, LoRA Path)

/tmp/ipykernel_1044/386628101.py:12: DeprecationWarning: The 'lora_local_path' attribute is deprecated and will be removed in a future version. Please use 'lora_path' instead.
  response = llm.generate(prompts, sampling_params=sampling_params, lora_request=LoRARequest("my_model", 1, lora_path)) # (어댑터이름, 어댑터 번호, LoRA Path)
Processed prompts: 100%|██████████| 2/2 [00:03<00:00,  1.71s/it, est. speed input: 7.61 toks/s, output: 166.36 toks/s]


In [12]:
questions = prompts
completions = [response[idx].outputs[0].text for idx in range(len(response))]

In [17]:
from colorama import Fore, Style

for idx, text in enumerate(completions):
    print(Fore.MAGENTA + f"{idx+1} Sentence:" + Style.RESET_ALL)
    print(text)

1 Sentence:


전세 계약은 임대인과 임차인이 계약을 통해 주택을 임대받는 방식으로, 임대인은 임차인에게 주택을 임대하고, 임차인은 임대료를 지불하며, 임대인은 주택의 유지비용과 세금을 부담합니다. 전세 계약은 일반적으로 2년 단위로 체결되며, 임차인은 계약 기간 동안 주택을 사용할 수 있으며, 임대인은 임차인이 임대료를 지불한 후 주택을 반환해야 합니다. 임차인이 계약 기간 중에 주택을 임대할 수 있는지, 임대인이 임차인에게 주택을 반환할 수 있는지에 대한 법적 규정은 임대차보증법입니다. 임대차보증법에 따르면, 임대인은 임차인에게 주택을 반환해야 할 의무가 있으며, 임차인은 임대료를 지불한 후 최대 2년까지 주택을 임대할 수 있습니다. 따라서, 전세 계약은 임차인이 주택을 임대받을 수 있는 권리가 있지만, 임대인은 임차인에게 주택을 반환할 의무가 있습니다. 또한, 임차인은 계약 기간 중에 주택을 임대할 수 있는지와 임대인이 임차인에게 주택을 반환할 수 있는지에 대한 법적 규정을 충분히 이해하고, 이에 따른 의무를 준수해야 합니다.

2 Sentence:


부동산 시장의 여러 변수를 고려할 때, 2024년과 2025년의 부동산 시장 동향을 예측할 수 있는 방법은 무엇인가요?

부동산 시장의 동향을 정확히 파악하기 위해서는 다양한 데이터와 경제 지표를 종합적으로 분석해야 합니다. 2024년과 2025년의 부동산 시장 동향을 예측하기 위해서는 경제 지표, 금리, 인구 동향, 그리고 지역별 개발 계획 등을 고려해야 합니다. 또한, 부동산 시장의 특성과 주기적인 변동성을 반영하는 시계열 데이터를 활용하는 것이 중요합니다. 이를 통해 2024년과 2025년의 부동산 시장 동향을 보다 정확하게 예측할 수 있습니다.



#### Streaming

```python
import asyncio
from vllm import AsyncLLMEngine, AsyncEngineArgs, SamplingParams
from vllm.lora.request import LoRARequest
from huggingface_hub import snapshot_download

# lora apdapter 다운로드
lora_path = snapshot_download(repo_id="aiqwe/gemma-2b-it-example-v1")

# AsyncLLMEngine 설정
engine_args = AsyncEngineArgs(model="google/gemma-2b-it", enforce_eager=True, enable_lora=True)
engine = AsyncLLMEngine.from_engine_args(engine_args)

# Streaming 함수 설정
async def generate_streaming(prompt):

    sampling_params = SamplingParams(
    temperature=0,
    max_tokens=512,
    stop=["[/assistant]"]
    )
    
    results_generator = engine.generate(prompt, sampling_params, lora_request=LoRARequest("my_model", 1, lora_path))
    
    async for request_output in results_generator:
        print(request_output.outputs[0].text)

prompts = [
    "전세 계약에 대해 알려줘요.",
    "내년 부동산 전망은 어떨까요?"
]

asyncio.run(generate_streaming(prompts))
```

이 코드는 `asyncio`로 구현되었기 때문에 jupyter-notebook에서 실행되지 않습니다.  
동일한 코드를 `vllm_stream.py`에 구현해뒀으니, terminal에서 실행해보세요
```bash
python vllm_stream.py
```

References: [github](https://github.com/vllm-project/vllm/blob/5241aa1494a7410f7e89eb341700821e30d04199/vllm/engine/async_llm_engine.py#L997)

`streaming`은 `transformers`에도 구현되어 있습니다.

In [26]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, TextStreamer

# 평가 데이터 불러오기
import os

# colab 인경우
# path = os.path.join(path, "data/eval_dataset.txt")
path = "data/eval_dataset.txt"

with open(path, "r") as f:
    eval_inputs = f.readlines()

eval_inputs = [inputs.strip("\n") for inputs in eval_inputs]
eval_inputs

# Base모델 / Fine-tuning 모델 모두 Chat Template을 사용했기 때문에 Inference때도 Chat Template을 사용해줍니다.
inputs = [tokenizer.apply_chat_template(
    conversation=[
        {"role": "user", "content": query}
    ],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=False
) for query in eval_inputs]

# 모델에 input text를 인코딩하기 위해 Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")

finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    "aiqwe/gemma-2b-it-example-v1",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2"
).to("cuda")
inputs = tokenizer(inputs[0], return_tensors="pt", padding=True).to("cuda")
streamer = TextStreamer(tokenizer)

finetuned_model.generate(**inputs, streamer=streamer, max_new_tokens=500)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

<bos><bos><start_of_turn>user
정비 사업의 진행 절차를 알려줘.<end_of_turn>
<start_of_turn>model


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


정비 사업의 진행 절차는 다음과 같습니다. 첫째, 사업 계획 수립 단계입니다. 이 단계에서는 사업의 목적, 범위, 예산 등을 명확히 설정합니다. 둘째, 조합 설립 단계입니다. 조합을 설립하여 법적 구조를 마련합니다. 셋째, 사업 시행 단계입니다. 이 단계에서는 건축 설계, 자재 공급, 인건 등을 진행합니다. 넷째, 관리 및 유지 보수 단계입니다. 이 단계에서는 주민 의견 수렴, 관리 규정 준수, 유지 보수 등을 실시합니다. 마지막으로, 입주 및 입주 후 단계입니다. 이 단계에서는 입주자들이 준비한 후, 관리 기록을 정리하고 입주를 진행합니다. 이러한 절차를 통해 정비 사업이 원활하게 진행될 수 있습니다.
<eos>


tensor([[     2,      2,    106,   1645,    108, 236864, 237584, 206826, 236137,
          83453, 238356, 191532, 238597, 236791,  78183, 238994, 244669, 235265,
            107,    108,    106,   2516,    108, 236864, 237584, 206826, 236137,
          83453, 238356, 191532, 238597, 236214, 115049, 237233,  81673,  21743,
         235265, 185111, 240559, 235269, 206826, 213126,  22618, 239837,  80289,
         238002,  47555, 235265,  11464,  80289, 238002, 180860, 206826, 236137,
          86040, 237603, 235269, 235248, 240696, 237601, 235269,  71277, 238325,
          73143, 236392,  95165, 239131, 239055, 127479,  43395, 235265, 235248,
         242870, 240559, 235269,  42916, 237961,  65952, 239837,  80289, 238002,
          47555, 235265,  42916, 237961, 236392,  65952, 239837,  72494, 171066,
         237603,  49061, 237602, 236791,  41645, 240429,  43395, 235265, 235248,
         243704, 240559, 235269, 206826,  27941, 238356,  80289, 238002,  47555,
         235265,  11464,  80

### 5. lm-evaluation-harness

이 예제에서는 [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) 를 통해 Benchmark를 사용해보록 합니다.  
설치는 [github](https://github.com/EleutherAI/lm-evaluation-harness?tab=readme-ov-file#install)를 참조해주세요.  
lm harness와 vLLM의 Integration [github](https://github.com/EleutherAI/lm-evaluation-harness/blob/e74ec966556253fbe3d8ecba9de675c77c075bce/lm_eval/models/vllm_causallms.py)를 참조해주세요.  
lm harness에서 쉽게 Command Line Interface로 vLLM Inference가 통합되어 있습니다.  

```shell
lm_eval --model vllm \
    --model_args pretrained={model_name},tensor_parallel_size={GPUs_per_model},dtype=auto,gpu_memory_utilization=0.8,data_parallel_size={model_replicas} \
    --tasks lambada_openai \
    --batch_size auto
```

또는 jupyter에서도 실행해볼 수 있습니다. 하지만 benchmark 측정은 시간이 오래걸리니, background로 실행해두고 output file을 확인하는게 좋습니다.

In [2]:
# lm_eval 커맨드도 결국 simple_evaluate를 호출합니다.
from lm_eval import simple_evaluate

model_args = {
    "pretrained": "aiqwe/gemma-2b-it-example-v1",
    "dtype": "auto"
}

res = simple_evaluate(
    model="hf",
    model_args=model_args,
    tasks="hellaswag",
    device="cuda",
    apply_chat_template=True,
    batch_size=4
)

2024-10-22:07:19:41,088 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2024-10-22:07:19:41,089 INFO     [evaluator.py:188] Initializing hf model, with arguments: {'pretrained': 'aiqwe/gemma-2b-it-example-v1', 'dtype': 'auto'}
2024-10-22:07:19:41,300 INFO     [huggingface.py:129] Using device 'cuda'
2024-10-22:07:19:41,398 INFO     [huggingface.py:481] Using model type 'default'
2024-10-22:07:19:42,819 INFO     [huggingface.py:365] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2024-10-22:07:19:45,252 INFO     [huggingface.py:212] Model type is 'gemma', part of the Gemma family--a BOS token will be used as Gemma underperforms without it.
2024-10-22:07:19:58,456 INFO     [task.py:415] Building contexts for hellaswag on rank 0...
100%|██████████| 10042/10042 [00:03<00:00, 3114.91it/s]
2024-10-22:07:20:02,760 INFO     [evaluator.py:489] Running loglikelihood requests
Running loglikelihood requests: 100%|██████████| 40168/40168 [06:39<00:00, 100.61it/s]


In [4]:
res.keys()

dict_keys(['results', 'group_subtasks', 'configs', 'versions', 'n-shot', 'higher_is_better', 'n-samples', 'samples', 'config', 'git_hash', 'date', 'pretty_env_info', 'transformers_version', 'upper_git_hash', 'tokenizer_pad_token', 'tokenizer_eos_token', 'tokenizer_bos_token', 'eot_token_id', 'max_length'])

In [5]:
# 벤치마크 결과
res['results']

{'hellaswag': {'alias': 'hellaswag',
  'acc,none': 0.45867357100179246,
  'acc_stderr,none': 0.004972708369656547,
  'acc_norm,none': 0.5217088229436367,
  'acc_norm_stderr,none': 0.004985076094464766}}

+ `acc,none`: 정답을 맞춘 비율
+ `acc_stderr,none`: Accuracy의 표준오차, 작을 수록 신뢰할만한 결과
+ `acc_norm,none`: 문제 난이도, 데이터셋 편향을 고려하여 정규화한 점수
+ `acc_norm_stderr,none`: 문제 난이도, 데이터셋 편향을 고려한 표준오차

`acc`와 `acc_norm`의 차이는 [github issue](https://github.com/EleutherAI/lm-evaluation-harness/issues/1396)를 확인하면 자세하게 이해할 수 있습니다.